# LIME for Explainability in Machine Learning
### *Chapter 3 — XAI Techniques | Explainable AI in Medical Systems*

---

**LIME** (Local Interpretable Model-agnostic Explanations) is one of the most widely used post-hoc XAI
frameworks, appearing in 18.3% of the 644 clinical papers reviewed in Chapter 4. Unlike SHAP, LIME is
**fully model-agnostic**: it treats any classifier as a black box and approximates its local behaviour
with a simple interpretable surrogate model.

### Core idea

For a given prediction, LIME:
1. **Perturbs** the input by generating nearby samples
2. **Queries** the black-box model on each perturbed sample
3. **Weights** each sample by its proximity to the original instance
4. **Fits a sparse linear model** (the surrogate) on the weighted neighbourhood
5. Returns the **coefficients** as the explanation

The result answers: *which features most influenced this specific prediction, and in which direction?*

---

## Contents

1. [Setup and overview](#1)
2. [LIME for tabular data — breast cancer classification](#2)
   - 2.1 LimeTabularExplainer setup
   - 2.2 Local explanation — single patient
   - 2.3 Comparing explanations across patients
   - 2.4 Aggregated quasi-global importance
   - 2.5 LIME vs. model feature importance
3. [LIME for text data — clinical note classification](#3)
   - 3.1 Dataset and model
   - 3.2 Word-level explanations
   - 3.3 Correct vs. ambiguous notes
4. [LIME for image data — medical image explanation](#4)
   - 4.1 Synthetic medical image and classifier
   - 4.2 Superpixel explanations
   - 4.3 Positive and negative regions
5. [LIME vs. SHAP: key differences](#5)
6. [Summary and clinical considerations](#6)


<a id='1'></a>
## 1. Setup and overview


In [ ]:
# Install required packages (run once)
# !pip install lime scikit-learn pandas numpy matplotlib seaborn scikit-image Pillow


In [ ]:
import lime
import lime.lime_tabular
import lime.lime_text
import lime.lime_image

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from skimage.segmentation import mark_boundaries

warnings.filterwarnings('ignore')
np.random.seed(42)
print('All packages imported successfully.')


### LIME explainer types

LIME provides three explainers, each designed for a specific data modality:

| Explainer | Input type | Perturbation strategy | Medical use case |
|---|---|---|---|
| `LimeTabularExplainer` | Structured / EHR data | Feature value sampling | Risk scores, lab values |
| `LimeTextExplainer` | Clinical text | Word/token removal | Notes, NLP, ICD coding |
| `LimeImageExplainer` | Images | Superpixel occlusion | X-ray, dermoscopy, pathology |

All three share the same underlying principle: **local surrogate fitting in a perturbed neighbourhood**.


<a id='2'></a>
## 2. LIME for tabular data — breast cancer classification

We use the Wisconsin Breast Cancer Dataset (569 patients, 30 features) to mirror the workflow most common
in medical XAI papers. The task is binary classification: **malignant (0)** vs. **benign (1)**.


In [ ]:
# ── Load and split dataset ────────────────────────────────────────────────────
data = load_breast_cancer()
X    = pd.DataFrame(data.data, columns=data.feature_names)
y    = pd.Series(data.target, name='diagnosis')  # 0=malignant, 1=benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')
print(f'Features     : {X_train.shape[1]}')


In [ ]:
# ── Train Random Forest ───────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print(f'Random Forest — AUC-ROC: {auc:.4f}')
print(classification_report(y_test, rf.predict(X_test),
                             target_names=['malignant', 'benign']))


### 2.1 LimeTabularExplainer setup

The tabular explainer needs the **training data** to estimate feature distributions for perturbation
sampling and determine discretisation thresholds for continuous features.

Key parameters:
- `discretize_continuous=True` — bins continuous features into quartiles, producing readable rules
  such as `worst radius > 18.5`
- `kernel_width` — controls locality decay; smaller = tighter neighbourhood
- `num_samples` in `explain_instance` — more samples give more stable but slower explanations


In [ ]:
# ── Initialise LimeTabularExplainer ──────────────────────────────────────────
explainer_tab = lime.lime_tabular.LimeTabularExplainer(
    training_data        = X_train.values,
    feature_names        = list(X_train.columns),
    class_names          = ['malignant', 'benign'],
    mode                 = 'classification',
    discretize_continuous= True,
    random_state         = 42
)
print('LimeTabularExplainer initialised.')
print(f'Classes : {explainer_tab.class_names}')


### 2.2 Local explanation — single patient

A local explanation answers: *why did the model predict this specific patient as malignant or benign?*

Each feature is assigned a **weight** (surrogate coefficient):
- **Positive weight** — feature value supports the predicted class
- **Negative weight** — feature value contradicts the predicted class

Discretised rules (e.g., `worst radius > 18.55`) make the explanation directly readable by clinicians.


In [ ]:
# ── Explain a single test instance ───────────────────────────────────────────
patient_idx = 3   # change this index to explore other patients
true_label  = 'benign' if y_test.iloc[patient_idx] == 1 else 'malignant'
pred_probs  = rf.predict_proba(X_test.iloc[[patient_idx]])[0]
pred_label  = 'benign' if pred_probs[1] > 0.5 else 'malignant'

print(f'Patient {patient_idx}')
print(f'  True label     : {true_label}')
print(f'  Predicted      : {pred_label}')
print(f'  P(malignant)   : {pred_probs[0]:.4f}')
print(f'  P(benign)      : {pred_probs[1]:.4f}')

exp = explainer_tab.explain_instance(
    data_row    = X_test.values[patient_idx],
    predict_fn  = rf.predict_proba,
    num_features= 10,
    num_samples = 1000
)

print(f'\nTop 10 LIME contributions (class = benign):')
for rule, weight in exp.as_list(label=1):
    direction = 'supports benign' if weight > 0 else 'supports malignant'
    print(f'  {weight:+.4f}  {rule:<45}  [{direction}]')


In [ ]:
# ── Visualise local explanation ───────────────────────────────────────────────
fig = exp.as_pyplot_figure(label=1)
plt.suptitle(
    f'LIME Local Explanation — Patient {patient_idx}\n'
    f'True: {true_label} | Predicted: {pred_label} | P(benign) = {pred_probs[1]:.3f}',
    fontsize=10, y=1.03
)
plt.tight_layout()
plt.show()


In [ ]:
print(f'Local surrogate R2 (fidelity in neighbourhood): {exp.score:.4f}')
print('Values close to 1.0 indicate the surrogate closely approximates the model locally.')
print()
print('Intercept and weight sum for the predicted class (benign=1):')
total = sum(w for _, w in exp.as_list(label=1))
print(f'  intercept      : {exp.intercept.get(1, 0):.4f}')
print(f'  sum(weights)   : {total:.4f}')
print(f'  approx output  : {exp.intercept.get(1, 0) + total:.4f}')


### 2.3 Comparing explanations across patients

A key property of LIME is that explanations are **local** — the same feature can contribute
differently for different patients with the same model. This is clinically meaningful: the model
may rely on different evidence for each individual.


In [ ]:
# ── Compare LIME explanations for 6 patients ─────────────────────────────────
n_patients = 6
n_features = 8
fig, axes  = plt.subplots(2, 3, figsize=(16, 10))
axes       = axes.flatten()

for i in range(n_patients):
    true_lbl = 'benign' if y_test.iloc[i] == 1 else 'malignant'
    pred_p   = rf.predict_proba(X_test.iloc[[i]])[0]
    pred_lbl = 'benign' if pred_p[1] > 0.5 else 'malignant'
    correct  = 'OK' if true_lbl == pred_lbl else 'WRONG'

    e = explainer_tab.explain_instance(
        X_test.values[i], rf.predict_proba,
        num_features=n_features, num_samples=500
    )
    rules, weights = zip(*e.as_list(label=1))
    colours = ['#2ecc71' if w > 0 else '#e74c3c' for w in weights]

    ax = axes[i]
    ax.barh(range(len(rules)), weights, color=colours)
    ax.set_yticks(range(len(rules)))
    ax.set_yticklabels([r[:35] for r in rules], fontsize=7)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(
        f'Patient {i} [{correct}]\nTrue: {true_lbl} | Pred: {pred_lbl} | P(B)={pred_p[1]:.2f}',
        fontsize=8
    )
    ax.set_xlabel('LIME weight', fontsize=7)

green_patch = mpatches.Patch(color='#2ecc71', label='Supports benign')
red_patch   = mpatches.Patch(color='#e74c3c', label='Supports malignant')
fig.legend(handles=[green_patch, red_patch], loc='lower center', ncol=2,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('LIME Local Explanations — 6 Patients (Random Forest)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()


### 2.4 Aggregated quasi-global importance

LIME is a local method, but aggregating weights across many patients produces a
**quasi-global** view of feature importance — common practice in medical XAI papers.

> **Important caveat:** Aggregated LIME weights are not directly comparable across instances
because each surrogate is fitted independently. Treat this as a qualitative, not quantitative,
global importance measure.


In [ ]:
# ── Aggregate LIME weights across 40 patients ─────────────────────────────────
n_aggregate = 40
all_weights = {}

for i in range(n_aggregate):
    e = explainer_tab.explain_instance(
        X_test.values[i], rf.predict_proba,
        num_features=10, num_samples=500
    )
    for rule, weight in e.as_list(label=1):
        # Extract base feature name from discretised rule
        feat = rule.split('<=')[0].split('>')[0].split('<')[0].strip()
        all_weights.setdefault(feat, []).append(abs(weight))

agg = pd.Series({k: np.mean(v) for k, v in all_weights.items()}).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
colours = plt.cm.Blues(np.linspace(0.4, 0.9, len(agg)))
ax.barh(range(len(agg)), agg.values, color=colours)
ax.set_yticks(range(len(agg)))
ax.set_yticklabels(agg.index, fontsize=9)
ax.set_xlabel('Mean |LIME weight| across patients', fontsize=10)
ax.set_title(
    f'Quasi-global LIME Feature Importance (n={n_aggregate} patients, RF)\n'
    '(qualitative — surrogate weights not directly comparable across instances)',
    fontsize=10, pad=10
)
plt.tight_layout()
plt.show()

print('Top 10 features by mean |LIME weight|:')
print(agg.sort_values(ascending=False).head(10).round(4).to_string())


### 2.5 LIME vs. model feature importance

Comparing LIME-aggregated importance with the model's built-in feature importance reveals
where the two diverge — potentially highlighting features the model uses globally but that
are locally unimportant for specific patients, or vice versa.


In [ ]:
# ── Compare LIME aggregate vs. RF built-in importance ────────────────────────
rf_imp   = pd.Series(rf.feature_importances_, index=X_train.columns)
lime_imp = pd.Series({k: np.mean(v) for k, v in all_weights.items()})

# Normalise both to [0, 1]
rf_norm   = rf_imp / rf_imp.max()
lime_norm = lime_imp / lime_imp.max()

# Top 12 features by RF importance
top12   = rf_norm.sort_values(ascending=False).head(12).index
comp_df = pd.DataFrame({
    'RF built-in'    : rf_norm[top12],
    'LIME aggregated': lime_norm.reindex(top12).fillna(0)
}).sort_values('RF built-in', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comp_df))
w = 0.35
ax.barh(x - w/2, comp_df['RF built-in'],       w, label='RF built-in', color='#2E75B6')
ax.barh(x + w/2, comp_df['LIME aggregated'],   w, label='LIME aggregated (normalised)', color='#E8A020')
ax.set_yticks(x)
ax.set_yticklabels(comp_df.index, fontsize=9)
ax.set_xlabel('Normalised importance score', fontsize=10)
ax.set_title('Feature Importance: RF built-in vs. LIME aggregated (normalised)', fontsize=11, pad=10)
ax.legend(fontsize=9)
ax.axvline(0, color='black', linewidth=0.6)
plt.tight_layout()
plt.show()


<a id='3'></a>
## 3. LIME for text data — clinical note classification

`LimeTextExplainer` explains text classifiers by iteratively **removing words** from the document
and observing how the prediction changes. Words whose removal most reduces the predicted probability
receive the highest importance weight.

We use a synthetic dataset of clinical note fragments classified as **Cardiology** or **Oncology** —
representative of real clinical NLP tasks such as ICD coding and specialty routing.


### 3.1 Dataset and model


In [ ]:
# ── Build synthetic clinical note dataset ─────────────────────────────────────
cardiac_notes = [
    'patient presents with chest pain radiating to left arm and shortness of breath',
    'severe chest tightness with diaphoresis and nausea typical of cardiac event',
    'myocardial infarction suspected based on chest pain and elevated troponin',
    'patient reports palpitations chest discomfort and irregular heartbeat',
    'ecg shows st elevation consistent with acute coronary syndrome',
    'heart failure symptoms including dyspnoea orthopnoea and peripheral oedema',
    'angina pectoris with exertional chest pain relieved by rest and nitrates',
    'atrial fibrillation with rapid ventricular rate and chest discomfort',
    'cardiomyopathy with reduced ejection fraction and exertional dyspnoea',
    'hypertensive crisis with headache elevated blood pressure and chest pain',
    'pericarditis with pleuritic chest pain and pericardial friction rub',
    'ventricular tachycardia episode managed with amiodarone and defibrillation',
]

oncology_notes = [
    'biopsy confirmed malignant tumour with lymph node involvement stage three',
    'breast cancer diagnosis following mammography and core needle biopsy',
    'lung adenocarcinoma with pleural effusion and metastatic spread confirmed',
    'colorectal cancer with elevated cea levels and colonoscopy findings',
    'patient undergoing chemotherapy for non-hodgkin lymphoma with remission',
    'prostate cancer with elevated psa and positive biopsy gleason score seven',
    'melanoma with breslow thickness greater than two millimetres lymph node biopsy',
    'ovarian cancer with ca125 elevation and peritoneal carcinomatosis',
    'glioblastoma multiforme resected with adjuvant radiotherapy and temozolomide',
    'pancreatic ductal adenocarcinoma with obstructive jaundice and weight loss',
    'hepatocellular carcinoma with alpha-fetoprotein elevation and liver cirrhosis',
    'cervical cancer staging with mri and positron emission tomography scan',
]

extras = ['', ' and fatigue', ' on examination', ' noted by clinician',
          ' requiring urgent assessment', ' with family history',
          ' after investigation', ' per clinical assessment']

def augment(templates, n=80):
    return [templates[i % len(templates)] + extras[i % len(extras)] for i in range(n)]

texts  = augment(cardiac_notes, 80) + augment(oncology_notes, 80)
labels = [0]*80 + [1]*80   # 0=Cardiology, 1=Oncology

X_txt_train, X_txt_test, y_txt_train, y_txt_test = train_test_split(
    texts, labels, test_size=0.20, random_state=42, stratify=labels
)
print(f'Training: {len(X_txt_train)} notes | Test: {len(X_txt_test)} notes')


In [ ]:
# ── Train TF-IDF + Logistic Regression pipeline ───────────────────────────────
text_pipeline = make_pipeline(
    TfidfVectorizer(max_features=500, ngram_range=(1, 2), stop_words='english'),
    LogisticRegression(max_iter=500, C=1.0, random_state=42)
)
text_pipeline.fit(X_txt_train, y_txt_train)

print(f'Text classifier accuracy: {text_pipeline.score(X_txt_test, y_txt_test):.4f}')
print(classification_report(y_txt_test, text_pipeline.predict(X_txt_test),
                             target_names=['Cardiology', 'Oncology']))


In [ ]:
# ── Initialise LimeTextExplainer ──────────────────────────────────────────────
explainer_text = lime.lime_text.LimeTextExplainer(
    class_names  = ['Cardiology', 'Oncology'],
    random_state = 42
)
print('LimeTextExplainer initialised.')


### 3.2 Word-level explanations

For text, LIME creates perturbed documents by randomly removing words and queries the model on each.
Words whose removal most reduces the predicted probability are assigned the highest importance.


In [ ]:
# ── Explain a clinical note ───────────────────────────────────────────────────
note_idx  = 0
note_text = X_txt_test[note_idx]
true_cls  = ['Cardiology', 'Oncology'][y_txt_test[note_idx]]
pred_prob = text_pipeline.predict_proba([note_text])[0]
pred_cls  = 'Cardiology' if pred_prob[0] > 0.5 else 'Oncology'

print(f'Note: {note_text}')
print(f'True class      : {true_cls}')
print(f'Predicted class : {pred_cls}')
print(f'P(Cardiology)   : {pred_prob[0]:.4f}')
print(f'P(Oncology)     : {pred_prob[1]:.4f}')

exp_text = explainer_text.explain_instance(
    note_text,
    text_pipeline.predict_proba,
    num_features = 8,
    num_samples  = 500,
    labels       = [0, 1]
)

print(f'\nKey words driving Oncology prediction (label=1):')
for word, weight in exp_text.as_list(label=1):
    direction = 'Oncology' if weight > 0 else 'Cardiology'
    print(f'  {weight:+.4f}  {word!r:<25}  -> {direction}')


In [ ]:
# ── Visualise word contributions for both classes ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label_idx, cls_name, colour in [
    (axes[0], 0, 'Cardiology', '#2E75B6'),
    (axes[1], 1, 'Oncology',   '#C0392B'),
]:
    words, weights = zip(*exp_text.as_list(label=label_idx))
    bar_colours = ['#2ecc71' if w > 0 else '#e74c3c' for w in weights]
    ax.barh(range(len(words)), weights, color=bar_colours)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels([f'{w!r}' for w in words], fontsize=9)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Word contributions -> {cls_name}', fontsize=10)
    ax.set_xlabel('LIME weight', fontsize=9)

fig.suptitle(
    f'LIME Text Explanation | True: {true_cls} | Predicted: {pred_cls}',
    fontsize=10, y=1.03
)
plt.tight_layout()
plt.show()


### 3.3 Correct vs. ambiguous notes

LIME explanations are valuable for **diagnosing difficult cases**: when a note contains terminology
from both specialties, the explanation reveals which words drove the model's decision —
useful for identifying systematic weaknesses in clinical NLP models.


In [ ]:
# ── Compare a clear-cut note vs. an ambiguous mixed-terminology note ──────────
clear_note     = X_txt_test[0]
ambiguous_note = ('patient has chest pain and elevated troponin '
                  'biopsy showed malignant cells and lymph node involvement')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, note, title_prefix in [
    (axes[0], clear_note,     'Clear-cut note'),
    (axes[1], ambiguous_note, 'Ambiguous (mixed terminology)'),
]:
    p = text_pipeline.predict_proba([note])[0]
    predicted = 'Cardiology' if p[0] > 0.5 else 'Oncology'

    e = explainer_text.explain_instance(
        note, text_pipeline.predict_proba, num_features=8, num_samples=500
    )
    words, weights = zip(*e.as_list(label=1))
    colours = ['#2ecc71' if w > 0 else '#e74c3c' for w in weights]

    ax.barh(range(len(words)), weights, color=colours)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels([f'{w!r}' for w in words], fontsize=8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(
        f'{title_prefix}\nPredicted: {predicted} (P(Onc)={p[1]:.2f})\n'
        f'Note: {note[:55]}...',
        fontsize=8
    )
    ax.set_xlabel('LIME weight -> Oncology', fontsize=8)

fig.suptitle('LIME Text: Clear vs. Ambiguous Clinical Note', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()


<a id='4'></a>
## 4. LIME for image data — medical image explanation

`LimeImageExplainer` partitions an image into **superpixels** (contiguous perceptually similar regions)
and iteratively occludes different subsets to identify which regions most influence the prediction.

This is applicable to medical imaging tasks including dermoscopy, chest X-ray, histopathology,
and retinal fundus image analysis.

We use a **synthetic medical-style image** with a bright upper-left quadrant simulating a
lesion/abnormality, classified by a toy model that predicts abnormality from regional brightness.


In [ ]:
# ── Create synthetic medical-style image ──────────────────────────────────────
np.random.seed(42)
img_size = 96

image = np.random.uniform(0.05, 0.20, (img_size, img_size, 3))

# Simulate lesion: bright upper-left quadrant
h, w = img_size // 2, img_size // 2
image[:h, :w, :] += np.random.uniform(0.50, 0.80, (h, w, 3))
image = np.clip(image, 0, 1)

# Add mild texture variation to normal regions
image[h:, w:, :] += np.random.uniform(0.00, 0.10, (h, w, 3))
image[:h, w:, :] += np.random.uniform(0.05, 0.15, (h, w, 3))
image = np.clip(image, 0, 1)

print(f'Image shape : {image.shape}')
print('Brightness by quadrant:')
print(f'  Upper-left  (lesion)  : {image[:h, :w, :].mean():.3f}')
print(f'  Upper-right (normal)  : {image[:h, w:, :].mean():.3f}')
print(f'  Lower-left  (normal)  : {image[h:, :w, :].mean():.3f}')
print(f'  Lower-right (normal)  : {image[h:, w:, :].mean():.3f}')

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image)
ax.add_patch(plt.Rectangle((0,0), w, h, fill=False, edgecolor='red', linewidth=2))
ax.text(5, 10, 'Lesion\nregion', color='red', fontsize=8, fontweight='bold')
ax.set_title('Synthetic medical image\n(red box = simulated lesion)', fontsize=9)
ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# ── Toy image classifier (black box) ─────────────────────────────────────────
# Represents a CNN that predicts abnormality based on brightness patterns.
# In practice: replace with rf.predict_proba, or a trained CNN.

def medical_image_classifier(images):
    results = []
    for img in images:
        h_i, w_i = img.shape[0] // 2, img.shape[1] // 2
        lesion_b  = img[:h_i, :w_i, :].mean()
        overall_b = img.mean()
        ratio      = lesion_b / (overall_b + 1e-6)
        p_abn      = float(np.clip((ratio - 1.0) / 2.0, 0, 1))
        results.append([1 - p_abn, p_abn])
    return np.array(results)

probs = medical_image_classifier([image])[0]
print(f'P(normal)   : {probs[0]:.4f}')
print(f'P(abnormal) : {probs[1]:.4f}')
print(f'Predicted   : {"Abnormal" if probs[1] > 0.5 else "Normal"}')


### 4.2 Superpixel explanations

LIME segments the image using the SLIC algorithm, then identifies which superpixels are
most influential. `get_image_and_mask()` retrieves the image with relevant boundaries overlaid.


In [ ]:
# ── Compute LIME image explanation ────────────────────────────────────────────
explainer_img = lime.lime_image.LimeImageExplainer(random_state=42)

print('Computing LIME image explanation (generating perturbed samples)...')
exp_img = explainer_img.explain_instance(
    image,
    medical_image_classifier,
    top_labels  = 2,
    hide_color  = 0,        # black occlusion for hidden superpixels
    num_samples = 500,
    num_features= 10,
    batch_size  = 50
)

predicted_label = exp_img.top_labels[0]
label_names     = {0: 'Normal', 1: 'Abnormal'}
print(f'Top predicted label : {label_names[predicted_label]}')


In [ ]:
# ── Visualise superpixel explanations ─────────────────────────────────────────
predicted_label = exp_img.top_labels[0]
lname           = label_names[predicted_label]

# All superpixels (positive=support, negative=contradict)
temp_all, mask_all = exp_img.get_image_and_mask(
    predicted_label, positive_only=False, num_features=10, hide_rest=False)

# Positive superpixels shown, rest visible
temp_pos, mask_pos = exp_img.get_image_and_mask(
    predicted_label, positive_only=True, num_features=5, hide_rest=False)

# Positive superpixels only, rest hidden
temp_foc, mask_foc = exp_img.get_image_and_mask(
    predicted_label, positive_only=True, num_features=5, hide_rest=True)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(image)
axes[0].set_title(f'Original\nPredicted: {lname} (P={probs[predicted_label]:.3f})', fontsize=9)
axes[0].axis('off')

axes[1].imshow(mark_boundaries(temp_all / (temp_all.max()+1e-6), mask_all))
axes[1].set_title('All superpixels\n(green=support, red=contradict)', fontsize=9)
axes[1].axis('off')

axes[2].imshow(mark_boundaries(temp_pos / (temp_pos.max()+1e-6), mask_pos))
axes[2].set_title('Supporting superpixels\n(rest visible)', fontsize=9)
axes[2].axis('off')

axes[3].imshow(mark_boundaries(temp_foc / (temp_foc.max()+1e-6), mask_foc))
axes[3].set_title('Supporting superpixels\n(rest hidden)', fontsize=9)
axes[3].axis('off')

fig.suptitle(f'LIME Image Explanation — Predicted class: {lname}', fontsize=11, y=1.03)
plt.tight_layout()
plt.show()


### 4.3 Positive and negative regions

Examining both classes reveals whether the model focuses on clinically relevant regions
and helps identify failure modes where the model relies on artefacts rather than pathology.


In [ ]:
# ── Explanations for both classes ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for row, (label_idx, lname) in enumerate([(1, 'Abnormal'), (0, 'Normal')]):
    t_all, m_all = exp_img.get_image_and_mask(
        label_idx, positive_only=False, num_features=8, hide_rest=False)
    t_pos, m_pos = exp_img.get_image_and_mask(
        label_idx, positive_only=True, num_features=5, hide_rest=False)
    t_neg, m_neg = exp_img.get_image_and_mask(
        label_idx, positive_only=False, negative_only=True, num_features=5, hide_rest=False)

    axes[row,0].imshow(mark_boundaries(t_all/(t_all.max()+1e-6), m_all))
    axes[row,0].set_title(f'Class: {lname}\nAll influential regions', fontsize=9)
    axes[row,0].axis('off')

    axes[row,1].imshow(mark_boundaries(t_pos/(t_pos.max()+1e-6), m_pos))
    axes[row,1].set_title(f'Supports -> {lname}', fontsize=9)
    axes[row,1].axis('off')

    axes[row,2].imshow(mark_boundaries(t_neg/(t_neg.max()+1e-6), m_neg))
    axes[row,2].set_title(f'Contradicts -> {lname}', fontsize=9)
    axes[row,2].axis('off')

fig.suptitle('LIME Image: Supporting vs. contradicting regions per class', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()


<a id='5'></a>
## 5. LIME vs. SHAP: key differences

Both LIME and SHAP are post-hoc, model-agnostic local explanation methods, but they differ
in ways that affect their suitability for different medical XAI applications.


In [ ]:
comparison = pd.DataFrame([
    {'Property':'Theoretical basis',    'LIME':'Local linear surrogate',          'SHAP':'Shapley values (game theory)'},
    {'Property':'Explanation values',   'LIME':'Surrogate coefficients (approx)','SHAP':'Fair attributions (exact for TreeSHAP)'},
    {'Property':'Additivity',           'LIME':'Not guaranteed to sum to f(x)',   'SHAP':'Guaranteed: base + sum(phi) = f(x)'},
    {'Property':'Consistency',          'LIME':'Stochastic — varies between runs','SHAP':'Deterministic for TreeSHAP'},
    {'Property':'Image support',        'LIME':'Yes — superpixel occlusion',      'SHAP':'Yes — GradientSHAP (CNN only)'},
    {'Property':'Text support',         'LIME':'Yes — word removal',              'SHAP':'Yes — via KernelSHAP (slow)'},
    {'Property':'Computational cost',   'LIME':'Medium (n_samples x model calls)','SHAP':'Low for TreeSHAP; high for KernelSHAP'},
    {'Property':'Neighbourhood tuning', 'LIME':'Requires kernel_width choice',    'SHAP':'Defined by Shapley axioms'},
    {'Property':'Best medical use',     'LIME':'Clinical text, imaging, any model','SHAP':'Tabular EHR, tree ensembles'},
    {'Property':'Literature share',     'LIME':'18.3% of 644 clinical papers',    'SHAP':'67.9% of 644 clinical papers'},
])
print(comparison.to_string(index=False))


<a id='6'></a>
## 6. Summary and clinical considerations

### Key properties demonstrated

1. **Locality:** The same feature can contribute differently for different patients — the model
   may rely on different evidence for each individual. This is clinically meaningful but requires
   careful communication to clinical audiences unfamiliar with local explanation methods.

2. **Model agnosticism:** LIME treats any classifier as a black box, making it uniquely suited
   to multi-modal settings and to models where no specialised explainer exists.

3. **Stochasticity:** Because LIME uses random sampling, explanations are not perfectly
   reproducible between runs. Always set a fixed `random_state` and validate stability by
   running multiple explanations for the same instance before clinical deployment.

4. **Interpretable rules:** Discretised continuous features produce rules (e.g.,
   `worst radius > 18.55`) directly readable by clinicians without statistical background.

5. **Local fidelity vs. global faithfulness:** The surrogate R2 score measures how well the
   linear surrogate approximates the black box in the local neighbourhood only. LIME makes
   no claims about global model behaviour.

### Clinical considerations

- **LIME explanations are not causal.** A word or feature that LIME identifies as influential
  drove the model's prediction, not necessarily the clinical outcome.
- **Neighbourhood stability** should be verified, especially for borderline predictions near
  the decision boundary where small perturbations can produce large explanation changes.
- **For imaging applications**, LIME superpixels should be compared against clinician-annotated
  ground truth to verify the model focuses on the correct anatomical region.

---

## Further reading

- Ribeiro, M.T., Singh, S. & Guestrin, C. (2016). *Why should I trust you? Explaining the
  predictions of any classifier.* KDD 2016. — The original LIME paper.
- Ribeiro, M.T., Singh, S. & Guestrin, C. (2018). *Anchors: High-precision model-agnostic
  explanations.* AAAI 2018. — Rule-based extension of LIME.
- LIME GitHub: https://github.com/marcotcr/lime
